In [ ]:
from playwright.async_api import async_playwright

playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=False)
page = await browser.new_page()
await page.goto("https://identity.walmart.com/account/login?client_id=5f3fb121-076a-45f6-9587-249f0bc160ff&redirect_uri=https%3A%2F%2Fwww.walmart.com%2Faccount%2FverifyToken&scope=openid+email+offline_access&tenant_id=elh9ie&state=%2F&code_challenge=TD9Ggyf8XzSAIDOBnQfmjNnegCq7EFKLQINxCsmCC4Y")

In [ ]:
from pathlib import Path

scan_js_path = "../tabvio/browser/scripts/scan-page.js"
js_script = Path(scan_js_path).read_text()


In [41]:
await page.main_frame.evaluate(js_script)

'{"url":"https://identity.walmart.com/account/login?client_id=5f3fb121-076a-45f6-9587-249f0bc160ff&redirect_uri=https%3A%2F%2Fwww.walmart.com%2Faccount%2FverifyToken&scope=openid+email+offline_access&tenant_id=elh9ie&state=%2F&code_challenge=TD9Ggyf8XzSAIDOBnQfmjNnegCq7EFKLQINxCsmCC4Y","title":"Login | Walmart","elements":[{"signature":"label||||Phone number or email (required)","tag":"label","text":"Phone number or email (required)","attrs":"","cx":588.3617630004883,"cy":259.23911333084106},{"signature":"input||Phone number or email (required)||Phone number or email (required)","tag":"input","text":"Phone number or email (required)","attrs":"type=text empty","cx":632.5465545654297,"cy":300.05821323394775},{"signature":"button|login-continue-button|||Continue","tag":"button","text":"Continue","attrs":"type=submit","cx":632.5465545654297,"cy":429.0372619628906},{"signature":"button||||Give feedback","tag":"button","text":"Give feedback","attrs":"type=button","cx":281.57607650756836,"cy"

In [ ]:
import re
from typing import Dict, List, Tuple

_WS = re.compile(r"\s+")


def _normalize(text: str) -> str:
    return _WS.sub(" ", text or "").strip()


def group_frames(frames: Dict[int, str]) -> List[Tuple[List[int], str]]:
    """Collapse frames sharing identical text, ordered by first frame index."""
    groups: Dict[str, List[int]] = {}
    for idx in sorted(frames):
        groups.setdefault(_normalize(frames[idx]), []).append(idx)
    return sorted(
        ((idxs, text) for text, idxs in groups.items()),
        key=lambda pair: pair[0][0],
    )

def fair_share(sizes: List[int], budget: int) -> List[int]:
    """Max-min fair allocation.

    Repeatedly hand out an equal share; any frame smaller than its share is
    settled at its true size and its leftover is redistributed to the frames
    still over-share. Means one 2000-char frame can't starve a 20-char one.
    """
    alloc = [0] * len(sizes)
    remaining, pending = budget, set(range(len(sizes)))
    while pending and remaining > 0:
        share = remaining // len(pending)
        if share == 0:
            break
        settled = {i for i in pending if sizes[i] <= share}
        if not settled:
            for i in pending:
                alloc[i] = share
            break
        for i in settled:
            alloc[i] = sizes[i]
            remaining -= sizes[i]
        pending -= settled
    return alloc

def drop_repeated_ngrams(text: str, n: int = 5) -> str:
    """Remove any word n-gram already seen earlier in this frame.

    All n words of a repeat are marked, so trailing fragments of a duplicated
    phrase disappear too instead of leaving debris behind.
    """
    words = text.split()
    if len(words) <= n:
        return text
    seen, drop = set(), [False] * len(words)
    for i in range(len(words) - n + 1):
        gram = tuple(w.lower() for w in words[i:i + n])
        if gram in seen:
            drop[i:i + n] = [True] * n
        else:
            seen.add(gram)
    return " ".join(w for w, d in zip(words, drop) if not d)

def head_tail(text: str, limit: int) -> str:
    if len(text) <= limit:
        return text
    head = int(limit * 0.45)
    tail = limit - head
    return (f"{text[:head].rstrip()} "
            f"…[{len(text) - limit} chars omitted]… "
            f"{text[-tail:].lstrip()}")

def digest(frames: Dict[int, str], char_budget: int = 1200) -> str:
    groups = group_frames(frames)
    filled = [(idxs, text) for idxs, text in groups if text]
    empty = [idxs for idxs, text in groups if not text]

    allocs = fair_share([len(text) for _, text in filled], char_budget)

    lines = []
    for (idxs, text), limit in zip(filled, allocs):
        if len(text) > limit:
            text = head_tail(drop_repeated_ngrams(text), max(limit, 40))
        label = ",".join(map(str, idxs))
        tag = f" (×{len(idxs)} identical)" if len(idxs) > 1 else ""
        lines.append(f"iframe[{label}]{tag} {text}")

    if empty:
        flat = ",".join(str(i) for idxs in empty for i in idxs)
        lines.append(f"iframe[{flat}] (empty)")

    return "\n".join(lines)

In [ ]:
import json

frames: Dict[int, str] = {}

for index, iframe in enumerate(page.frames):
    page_text = json.loads(await iframe.evaluate(js_script))['pageText']
    frames[index] = page_text

In [ ]:
digest(frames, char_budget=500)

In [ ]:
await browser.close()
await playwright.stop()
